<a href="https://www.kaggle.com/code/saibhossain/building-vanilla-elman-rnn-from-scratch?scriptVersionId=324669508" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 1. What Problem Does an RNN Solve?

A standard **Multi-Layer Perceptron (MLP)** processes data statically:
<span style="color:red">$$\mathbf{x} \rightarrow \mathbf{y}$$</span>

However, language is inherently sequential. For example, in the sentence:
> *"<span style="color:blue">I love deep learning </span>"*

The meaning of **"learning"** depends heavily on the preceding word **"deep"**. A standard feedforward network cannot naturally remember these previous tokens because it lacks memory.

An RNN solves this by introducing a **Hidden State (Memory)**. At every timestep $t$, the hidden state updates as follows:

<span style="color:red">$$h_t = f(x_t, h_{t-1})$$</span>

**Core concept:** 
Current understanding $=$ <span style="color:green">Current input</span> $+$ <span style="color:orange">Previous memory</span>. 

This recurrence mechanism allows information to persist across sequential data.



# 2. Elman RNN Architecture

The classic Elman RNN consists of three main structural layers:
* <span style="color:#3CAEA3">**Input layer**</span> ($x_t$)
* <span style="color:#F6D55C">**Hidden recurrent layer**</span> ($h_t$)
* <span style="color:#ED553B">**Output layer**</span> ($y_t$)

At each timestep $t$, the network executes the following mathematical operations sequentially:

#### <span style="color:#F6D55C">Step 1: Update the Hidden State</span>
The current hidden state mixes the new input with the previous memory through a $tanh$ activation function:
$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

#### <span style="color:#ED554B">Step 2: Generate the Raw Output</span>
The network projects the updated hidden state to the output space:
$$y_t = W_{hy} h_t + b_y$$

#### <span style="color:#3CAEA4">Step 3: Calculate Probabilities</span>
A softmax function normalizes the raw outputs into predictions (next-word probabilities):
$$p_t = \text{softmax}(y_t)$$


# 3. Why Training an RNN is Difficult

In a standard Multi-Layer Perceptron (MLP), the loss depends only on the current static input. 

However, in an RNN, the network carries a history. The final loss at any point depends on a deep chain of dependencies:
* <span style="color:#0275D8">**Current state**</span> ($h_t$)
* <span style="color:#F0AD4E">**Previous states**</span> ($h_{t-1}$)
* <span style="color:#5CB85C">**Previous-previous states**</span> ($h_{t-2}$)
* <span style="color:#5BC0DE">**The entire historical sequence**</span> ($h_0 \dots h_t$)

Because of this deep chain, gradients cannot just flow backward through layers; they must flow backward through **time layers**. 

This specialized algorithm is known as:
### <span style="color:#D9534F">BPTT — Backpropagation Through Time</span>

$$\frac{\partial L}{\partial W} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial W}$$

Every single weight update requires unrolling the network across the entire length of the input sequence, making training computationally heavy and prone to instability.


# 4. The Vanishing & Exploding Gradient Problem

During Backpropagation Through Time (BPTT), the network calculates derivatives across many timesteps. This forces the algorithm into a chain of **repeated matrix multiplications**:

$$\frac{\partial h_t}{\partial h_{t-1}}$$

Depending on the magnitude of the weights ($W_{hh}$) and the activation function derivatives, this continuous multiplication forces the gradients to scale exponentially:

* <span style="color:#D9534F">**Shrink $\rightarrow$ Vanish:**</span> If the values are $< 1$, gradients drop to zero. The network **forgets long-term dependencies**.
* <span style="color:#FFAD00">**Grow $\rightarrow$ Explode:**</span> If the values are $> 1$, gradients skyrocket. This causes **numerical instability (NaN errors)**.

---

### <span style="color:#5CB85C">💡 The Architectural Solution</span>

Standard RNNs cannot natively hold onto information over long sequences due to this math block. This critical flaw is exactly why more advanced gated architectures were invented:

1. <span style="color:#0275D8">**LSTM**</span> (Long Short-Term Memory)
2. <span style="color:#5BC0DE">**GRU**</span> (Gated Recurrent Unit)

---

<div class="alert alert-block alert-warning">
⚠️ <b>Interview Favorite Question:</b> Interviewers frequently ask candidates to explain <i>why</i> standard RNNs fail on long sequences and exactly how the math of the $$\frac{\partial h_t}{\partial h_{t-1}}$$ chain causes vanishing gradients.
</div>

### <span style="color:#D9534F">5. Deep Dive: The Mathematics of Vanishing Gradients</span>

The Vanishing Gradient Problem is the ultimate **Achilles' heel** of vanilla RNNs. During Backpropagation Through Time (BPTT), we unroll the network and calculate gradients backwards from the final time step $T$ down to the initial time step $1$.

Because the network shares the **exact same weight matrix** <span style="color:#F0AD4E">$W_{hh}$</span> across all time steps, backpropagating forces us to multiply the gradient by $W_{hh}$ and the derivative of the $\tanh$ activation function repeatedly.

---

#### <span style="color:#0275D8">📐 The Mathematical Proof (Chain Rule)</span>

By applying the chain rule, the gradient of the loss $L$ with respect to a hidden state far in the past ($h_1$) requires computing a product of Jacobians:

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_T} \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

Expanding the internal derivative $\frac{\partial h_t}{\partial h_{t-1}}$ yields:

$$\frac{\partial h_t}{\partial h_{t-1}} = \text{diag}\left(1 - \tanh^2(W_{xh}x_t + W_{hh}h_{t-1} + b_h)\right) \cdot W_{hh}$$

---

#### <span style="color:#222">📉 The Eigenvalue Effect</span>

The behavior of the long-term gradient depends heavily on the **largest eigenvalue** (spectral radius) of the shared weight matrix $W_{hh}$:

* <span style="color:#D9534F">**Vanishing ($\lambda_{max} < 1$):**</span> Multiplying $W_{hh}$ by itself $T$ times causes the gradient to **shrink exponentially toward zero**. The network "forgets" distant dependencies because the error signal vanishes before it can update early weights.
* <span style="color:#FFAD00">**Exploding ($\lambda_{max} > 1$):**</span> The gradients grow exponentially toward infinity. We typically mitigate this specific issue using **gradient clipping**.


In [1]:
import numpy as np

text = "Shall I compare thee to a summer's day?\nThou art more lovely and more temperate."
print(text,"\n")
chars = list(set(text))
print(chars,"\n")
vocab_size = len(chars)
print(vocab_size,"\n")

Shall I compare thee to a summer's day?
Thou art more lovely and more temperate. 

['.', 'e', 'T', 'h', 'a', 's', 'm', 'r', 'd', 'o', 'S', '\n', 'c', 'n', 'l', 'I', 'p', 'u', 'v', 'y', ' ', "'", 't', '?'] 

24 



In [2]:
char_to_ix = {ch: i for i, ch in enumerate(chars)}
char_to_ix

{'.': 0,
 'e': 1,
 'T': 2,
 'h': 3,
 'a': 4,
 's': 5,
 'm': 6,
 'r': 7,
 'd': 8,
 'o': 9,
 'S': 10,
 '\n': 11,
 'c': 12,
 'n': 13,
 'l': 14,
 'I': 15,
 'p': 16,
 'u': 17,
 'v': 18,
 'y': 19,
 ' ': 20,
 "'": 21,
 't': 22,
 '?': 23}

In [3]:
ix_to_char = {i: ch for i, ch in enumerate(chars)}
ix_to_char

{0: '.',
 1: 'e',
 2: 'T',
 3: 'h',
 4: 'a',
 5: 's',
 6: 'm',
 7: 'r',
 8: 'd',
 9: 'o',
 10: 'S',
 11: '\n',
 12: 'c',
 13: 'n',
 14: 'l',
 15: 'I',
 16: 'p',
 17: 'u',
 18: 'v',
 19: 'y',
 20: ' ',
 21: "'",
 22: 't',
 23: '?'}

In [4]:
hidden_size = 64      # Size of the hidden layer (memory)
seq_length = 10       # How many steps we unroll for BPTT
learning_rate = 0.1

In [5]:
# 3. Model Parameters (Initialized randomly)
Wxh = np.random.randn(hidden_size, vocab_size) * 0.01  # Input -> Hidden
Whh = np.random.randn(hidden_size, hidden_size) * 0.01 # Hidden -> Hidden
Why = np.random.randn(vocab_size, hidden_size) * 0.01  # Hidden -> Output
bh = np.zeros((hidden_size, 1))                        # Hidden bias
by = np.zeros((vocab_size, 1))                         # Output bias

In [6]:
def forward_backward(inputs, targets, hprev):
    """
    inputs, targets: lists of integers
    hprev: Hx1 array of initial hidden state
    Returns: loss, gradients on parameters, and last hidden state
    """
    # Dictionaries to store variables at each time step
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0
    
    # --- FORWARD PASS ---
    for t in range(len(inputs)):
        # One-hot encode the input character
        xs[t] = np.zeros((vocab_size, 1))
        xs[t][inputs[t]] = 1
        
        # Calculate hidden state: h_t = tanh(Wxh * x_t + Whh * h_{t-1} + bh)
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[-1 if t==0 else t-1]) + bh)
        
        # Calculate unnormalized log probabilities: y_t = Why * h_t + by
        ys[t] = np.dot(Why, hs[t]) + by
        
        # Softmax for probabilities
        ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
        
        # Cross-entropy loss
        loss += -np.log(ps[t][targets[t], 0])
        
    # --- BACKWARD PASS (BPTT) ---
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dhnext = np.zeros_like(hs[0])
    
    for t in reversed(range(len(inputs))):
        # Derivative of the loss with respect to output y
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1 # Backprop through softmax
        
        # Gradients for Why and by
        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        
        # Backprop into hidden state (gradient from output + gradient from next time step)
        dh = np.dot(Why.T, dy) + dhnext
        
        # Backprop through tanh nonlinearity (derivative of tanh(x) is 1 - tanh(x)^2)
        dhraw = (1 - hs[t] * hs[t]) * dh 
        
        # Gradients for bh, Wxh, Whh
        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)
        
        # Pass gradient backwards to the next time step (which is physically the previous step t-1)
        dhnext = np.dot(Whh.T, dhraw)
        
    # Gradient clipping to prevent exploding gradients (Crucial for vanilla RNNs!)
    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam)
        
    return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

In [7]:
def sample(h, seed_ix, n):
    """
    Generate a sequence of n characters.
    h: current hidden state
    seed_ix: starting character index
    """
    x = np.zeros((vocab_size, 1))
    x[seed_ix] = 1
    ixes = []
    
    for _ in range(n):
        h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
        y = np.dot(Why, h) + by
        p = np.exp(y) / np.sum(np.exp(y))
        
        # Sample a character index based on the probability distribution
        ix = np.random.choice(range(vocab_size), p=p.ravel())
        x = np.zeros((vocab_size, 1))
        x[ix] = 1
        ixes.append(ix)
        
    return ''.join(ix_to_char[ix] for ix in ixes)

# --- TRAINING LOOP ---
n, p = 0, 0
# Memory variables for Adagrad optimizer
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by)

# Initial hidden state
hprev = np.zeros((hidden_size, 1))

# Train for 5000 iterations
for n in range(5000):
    # Reset data pointer if we reach the end of the text
    if p + seq_length + 1 >= len(text) or n == 0:
        hprev = np.zeros((hidden_size, 1))
        p = 0
        
    # Grab a sequence of inputs and targets
    inputs = [char_to_ix[ch] for ch in text[p:p+seq_length]]
    targets = [char_to_ix[ch] for ch in text[p+1:p+seq_length+1]]
    
    # Forward and backward pass
    loss, dWxh, dWhh, dWhy, dbh, dby, hprev = forward_backward(inputs, targets, hprev)
    
    # Perform parameter update with Adagrad
    for param, dparam, mem in zip([Wxh, Whh, Why, bh, by], 
                                  [dWxh, dWhh, dWhy, dbh, dby], 
                                  [mWxh, mWhh, mWhy, mbh, mby]):
        mem += dparam * dparam
        param += -learning_rate * dparam / np.sqrt(mem + 1e-8)
        
    p += seq_length
    
    # Print progress
    if n % 1000 == 0:
        print(f"Iteration {n}, Loss: {loss:.4f}")
        print(f"Sample: '{sample(hprev, inputs[0], 50)}'\n")

Iteration 0, Loss: 31.7813
Sample: 'dp.eanrl TehdI.?.oSmhcpthaI'h.tev.?la?yus Tdu
?d.m'

Iteration 1000, Loss: 0.1514
Sample: 'ovely and more tu a summer's day?
Thou art more lo'

Iteration 2000, Loss: 0.0732
Sample: 'and more to a summer's day?
Thou art more loveay a'

Iteration 3000, Loss: 0.0318
Sample: 're lovely and more to a summer's day?
Thou art mor'

Iteration 4000, Loss: 0.0377
Sample: 'lou art more lovely and more to a summer's day?
Th'



### <span style="color:#D9534F">Breakdown: The Short-Term Memory Illusion</span>

When you look at a trained Vanilla RNN, it might seem smart because it successfully masters **short-term patterns** (like localized grammar and spelling). 

For instance, the model confidently predicts:
* The character `a` $\rightarrow$ should be followed by `n` $\rightarrow$ then `d`.
* The word `and` $\rightarrow$ is usually followed by a space and the word `more`.

#### <span style="color:#D9534F">❌ The Structural Amnesia</span>

However, the network has completely forgotten the **global context** of the input sequence. It might be trying to complete a poem, but it has totally forgotten that the poem is structurally required to end with the word <span style="color:#D9534F">**"temperate."**</span>

Because of the **Vanishing Gradient Problem**, Vanilla RNNs act exactly like someone suffering from severe short-term memory loss. 

---

#### <span style="color:#F0AD4E">🕰️ Why the Signal Disappears In Time</span>

When the network makes a mistake at the end of a long sentence, it triggers Backpropagation Through Time (BPTT) to pass an **"error signal" (the gradient)** backward through the timeline to update its weights:

1. The network assesses the error at time step $T$.
2. It attempts to pass this signal back to time step $1$.
3. <span style="color:#D9534F">**The Bottleneck:**</span> Every single time that error signal steps back by just one character, it gets multiplied by the shared weight matrix <span style="color:#F0AD4E">$W_{hh}$</span>.

$$\text{Gradient Scale} \propto (W_{hh})^T$$

If $W_{hh}$ contains fractional values, **multiplying a fraction by a fraction repeatedly shrinks the value toward zero.** 

By the time this vital error signal travels all the way back to the beginning of the sentence, it has **completely vanished**. Because no signal arrives, the network literally cannot calculate or learn how the beginning of a sentence structurally impacts the end of it.


# 5. Limitations of vanilla RNNs

### <span style="color:#0275D8">Short-Term Patterns vs. Long-Term Amnesia</span>

A vanilla RNN is excellent at capturing **short-range, local dependencies**, but completely falls apart when required to maintain a long-term memory buffer.

---

#### <span style="color:#5CB85C">✅ What a Vanilla RNN Learns Well</span>

Because gradients flow effectively over small distances, the network easily captures **local patterns** and statistical sub-structures, such as:
* <span style="color:#5CB85C">**Nearby characters**</span> (predicting spelling sequences)
* <span style="color:#5BC0DE">**Small words**</span> (basic syntactic structures)
* <span style="color:#F0AD4E">**Local suffixes/prefixes**</span> (word endings)

**Example of local syllable bonding:**
> `Th` $\rightarrow$ `ou` $\rightarrow$ `ly` $\rightarrow$ `ing`

The network learns these combinations effortlessly because the timesteps are physically close together in the unrolled sequence.

---

#### <span style="color:#D9534F">❌ The Fundamental Limitation</span>

As the input sequence grows longer, the hidden state acts like a crowded funnel. Every new token shifts and overwrites older representations. Because the gradient vanishes over long temporal spans, the RNN **completely forgets old context** from earlier parts of the text, rendering it useless for complex document processing or long-term dialogue context.


In [8]:
def test_vanishing_gradient(seq_len=30):
    print(f"\n--- RUNNING VANISHING GRADIENT TEST ({seq_len} steps) ---")
    
    # 1. Create a dummy sequence of characters
    test_inputs = [char_to_ix[chars[i % vocab_size]] for i in range(seq_len)]
    test_targets = [char_to_ix[chars[(i+1) % vocab_size]] for i in range(seq_len)]
    
    # Dictionaries for forward pass
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.zeros((hidden_size, 1))
    
    # --- FORWARD PASS ---
    for t in range(seq_len):
        xs[t] = np.zeros((vocab_size, 1))
        xs[t][test_inputs[t]] = 1
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[-1 if t==0 else t-1]) + bh)
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
        
    # --- BACKWARD PASS (Tracking the Gradient) ---
    dhnext = np.zeros_like(hs[0])
    
    print("Time Step (Backward) | Gradient Magnitude (L2 Norm)")
    print("-" * 55)
    
    # We step backwards from the end of the sequence to the beginning
    for t in reversed(range(seq_len)):
        dy = np.copy(ps[t])
        dy[test_targets[t]] -= 1 
        
        # Calculate the gradient for the hidden state
        dh = np.dot(Why.T, dy) + dhnext
        dhraw = (1 - hs[t] * hs[t]) * dh 
        
        # Pass it further back
        dhnext = np.dot(Whh.T, dhraw)
        
        # Calculate the magnitude (L2 Norm) of the gradient signal
        grad_magnitude = np.sqrt(np.sum(dhnext ** 2))
        
        # Print the magnitude every 5 steps
        if t % 5 == 0 or t == seq_len - 1:
            print(f"Step {t:02d} (t-{seq_len - t - 1:02d})       | {grad_magnitude:.10f}")

test_vanishing_gradient()


--- RUNNING VANISHING GRADIENT TEST (30 steps) ---
Time Step (Backward) | Gradient Magnitude (L2 Norm)
-------------------------------------------------------
Step 29 (t-00)       | 3.0299313051
Step 25 (t-04)       | 10.6656822402
Step 20 (t-09)       | 9.9966901113
Step 15 (t-14)       | 20.6954545383
Step 10 (t-19)       | 73.9514648514
Step 05 (t-24)       | 28.9318893097
Step 00 (t-29)       | 127.2192724362


# Big data 

In [ ]:
import numpy as np

text = """
Albert Einstein was born at Ulm, in Württemberg, Germany, on March 14, 1879.\n
Six weeks later the family moved to Munich, where he later on began his schooling at the Luitpold Gymnasium.\n
Later, they moved to Italy and Albert continued his education at Aarau, Switzerland and in 1896 he entered the Swiss Federal Polytechnic School in Zurich to be trained as a teacher in physics and mathematics.\n
In 1901, the year he gained his diploma, he acquired Swiss citizenship and, as he was unable to find a teaching post, he accepted a position as technical assistant in the Swiss Patent Office.\n
In 1905 he obtained his doctor’s degree.\n
During his stay at the Patent Office, and in his spare time, he produced much of his remarkable work and in 1908 he was appointed Privatdozent in Berne.\n
In 1909 he became Professor Extraordinary at Zurich, in 1911 Professor of Theoretical Physics at Prague, returning to Zurich in the following year to fill a similar post.\n
In 1914 he was appointed Director of the Kaiser Wilhelm Physical Institute and Professor in the University of Berlin. He became a German citizen in 1914 and remained in Berlin until 1933 when he renounced his citizenship for political reasons and emigrated to America to take the position of Professor of Theoretical Physics at Princeton*. He became a United States citizen in 1940 and retired from his post in 1945.\n
After World War II, Einstein was a leading figure in the World Government Movement, he was offered the Presidency of the State of Israel, which he declined, and he collaborated with Dr. Chaim Weizmann in establishing the Hebrew University of Jerusalem.\n
Einstein always appeared to have a clear view of the problems of physics and the determination to solve them.\n
He had a strategy of his own and was able to visualize the main stages on the way to his goal.\n
He regarded his major achievements as mere stepping-stones for the next advance.\n
At the start of his scientific work, Einstein realized the inadequacies of Newtonian mechanics and his special theory of relativity stemmed from an attempt to reconcile the laws of mechanics with the laws of the electromagnetic field.\n
He dealt with classical problems of statistical mechanics and problems in which they were merged with quantum theory: this led to an explanation of the Brownian movement of molecules. He investigated the thermal properties of light with a low radiation density and his observations laid the foundation of the photon theory of light.\n
In his early days in Berlin, Einstein postulated that the correct interpretation of the special theory of relativity must also furnish a theory of gravitation and in 1916 he published his paper on the general theory of relativity. During this time he also contributed to the problems of the theory of radiation and statistical mechanics.\n
In the 1920s, Einstein embarked on the construction of unified field theories, although he continued to work on the probabilistic interpretation of quantum theory, and he persevered with this work in America. He contributed to statistical mechanics by his development of the quantum theory of a monatomic gas and he has also accomplished valuable work in connection with atomic transition probabilities and relativistic cosmology.\n
After his retirement he continued to work towards the unification of the basic concepts of physics, taking the opposite approach, geometrisation, to the majority of physicists. Einstein’s researches are, of course, well chronicled and his more important works include Special Theory of Relativity (1905), Relativity (English translations, 1920 and 1950), General Theory of Relativity (1916), Investigations on Theory of Brownian Movement (1926), and The Evolution of Physics (1938).\n
Among his non-scientific works, About Zionism (1930), Why War? (1933), My Philosophy (1934), and Out of My Later Years (1950) are perhaps the most important.\n
Albert Einstein received honorary doctorate degrees in science, medicine and philosophy from many European and American universities. During the 1920’s he lectured in Europe, America and the Far East, and he was awarded Fellowships or Memberships of all the leading scientific academies throughout the world. He gained numerous awards in recognition of his work, including the Copley Medal of the Royal Society of London in 1925, and the Franklin Medal of the Franklin Institute in 1935.\n
Einstein’s gifts inevitably resulted in his dwelling much in intellectual solitude and, for relaxation, music played an important part in his life.\n
He married Mileva Maric in 1903 and they had a daughter and two sons; their marriage was dissolved in 1919 and in the same year he married his cousin, Elsa Löwenthal, who died in 1936. He died on April 18, 1955 at Princeton, New Jersey.
"""

# Unique characters
chars = sorted(list(set(text)))

# Vocabulary size
vocab_size = len(chars)

print("Total Characters :", len(text))
print("Vocabulary Size  :", vocab_size)

In [35]:
# Character mappings
char_to_ix = {ch: i for i, ch in enumerate(chars)}
print(char_to_ix)
ix_to_char = {i: ch for i, ch in enumerate(chars)}

{'\n': 0, ' ': 1, '(': 2, ')': 3, '*': 4, ',': 5, '-': 6, '.': 7, '0': 8, '1': 9, '2': 10, '3': 11, '4': 12, '5': 13, '6': 14, '7': 15, '8': 16, '9': 17, ':': 18, ';': 19, '?': 20, 'A': 21, 'B': 22, 'C': 23, 'D': 24, 'E': 25, 'F': 26, 'G': 27, 'H': 28, 'I': 29, 'J': 30, 'K': 31, 'L': 32, 'M': 33, 'N': 34, 'O': 35, 'P': 36, 'R': 37, 'S': 38, 'T': 39, 'U': 40, 'W': 41, 'Y': 42, 'Z': 43, 'a': 44, 'b': 45, 'c': 46, 'd': 47, 'e': 48, 'f': 49, 'g': 50, 'h': 51, 'i': 52, 'j': 53, 'k': 54, 'l': 55, 'm': 56, 'n': 57, 'o': 58, 'p': 59, 'q': 60, 'r': 61, 's': 62, 't': 63, 'u': 64, 'v': 65, 'w': 66, 'x': 67, 'y': 68, 'z': 69, 'ö': 70, 'ü': 71, '’': 72}


In [ ]:
hidden_size = 128      # RNN memory size
seq_length = 50        # BPTT unroll steps
learning_rate = 0.01
iterations = 20000

Wxh = np.random.randn(hidden_size, vocab_size) * 0.01  # Input -> Hidden
Whh = np.random.randn(hidden_size, hidden_size) * 0.01 # Hidden -> Hidden
Why = np.random.randn(vocab_size, hidden_size) * 0.01  # Hidden -> Output

# Biases
bh = np.zeros((hidden_size, 1))
by = np.zeros((vocab_size, 1))

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

In [34]:
def forward_backward(inputs, targets, hprev):

    xs, hs, ys, ps = {}, {}, {}, {}

    # Store previous hidden state
    hs[-1] = np.copy(hprev)

    loss = 0

    # =====================================================
    # FORWARD PASS
    # =====================================================

    for t in range(len(inputs)):

        # One-hot encoding
        xs[t] = np.zeros((vocab_size, 1))
        xs[t][inputs[t]] = 1

        # Hidden state
        hs[t] = np.tanh(
            np.dot(Wxh, xs[t]) +
            np.dot(Whh, hs[t - 1]) +
            bh
        )

        # Output logits
        ys[t] = np.dot(Why, hs[t]) + by

        # Probabilities
        ps[t] = softmax(ys[t])

        # Cross entropy loss
        loss += -np.log(ps[t][targets[t], 0])

    # =====================================================
    # BACKWARD PASS (BPTT)
    # =====================================================

    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)

    dbh = np.zeros_like(bh)
    dby = np.zeros_like(by)

    # Gradient flowing backward through time
    dhnext = np.zeros_like(hs[0])

    # Reverse through time
    for t in reversed(range(len(inputs))):

        # Softmax gradient
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1

        # Output layer gradients
        dWhy += np.dot(dy, hs[t].T)
        dby += dy

        # Gradient into hidden state
        dh = np.dot(Why.T, dy) + dhnext

        # tanh derivative
        dhraw = (1 - hs[t] * hs[t]) * dh

        # Hidden gradients
        dbh += dhraw

        dWxh += np.dot(dhraw, xs[t].T)

        dWhh += np.dot(dhraw, hs[t - 1].T)

        # Pass gradient backward through time
        dhnext = np.dot(Whh.T, dhraw)

    # =====================================================
    # GRADIENT CLIPPING
    # Prevent exploding gradients
    # =====================================================

    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam)

    return (
        loss,
        dWxh,
        dWhh,
        dWhy,
        dbh,
        dby,
        hs[len(inputs) - 1]
    )

# =========================================================
# 6. TEXT SAMPLING
# Generate text from trained RNN
# =========================================================

def sample(h, seed_ix, n):

    # Initial input
    x = np.zeros((vocab_size, 1))
    x[seed_ix] = 1

    generated_chars = []

    for _ in range(n):

        # Hidden state
        h = np.tanh(
            np.dot(Wxh, x) +
            np.dot(Whh, h) +
            bh
        )

        # Output
        y = np.dot(Why, h) + by

        # Probabilities
        p = softmax(y)

        # Sample next character
        ix = np.random.choice(range(vocab_size), p=p.ravel())

        # Prepare next input
        x = np.zeros((vocab_size, 1))
        x[ix] = 1

        generated_chars.append(ix_to_char[ix])

    return ''.join(generated_chars)

# =========================================================
# 7. ADAGRAD MEMORY
# =========================================================

mWxh = np.zeros_like(Wxh)
mWhh = np.zeros_like(Whh)
mWhy = np.zeros_like(Why)

mbh = np.zeros_like(bh)
mby = np.zeros_like(by)

# =========================================================
# 8. TRAINING LOOP
# =========================================================

n = 0
p = 0

# Initial hidden state
hprev = np.zeros((hidden_size, 1))

print("\nTraining Started...\n")

while n < iterations:

    # Reset at end of data
    if p + seq_length + 1 >= len(text):

        hprev = np.zeros((hidden_size, 1))
        p = 0

    # Input sequence
    inputs = [
        char_to_ix[ch]
        for ch in text[p:p + seq_length]
    ]

    # Target sequence (shifted by 1)
    targets = [
        char_to_ix[ch]
        for ch in text[p + 1:p + seq_length + 1]
    ]

    # Forward + Backward
    (
        loss,
        dWxh,
        dWhh,
        dWhy,
        dbh,
        dby,
        hprev
    ) = forward_backward(inputs, targets, hprev)

    # =====================================================
    # ADAGRAD PARAMETER UPDATE
    # =====================================================

    for param, dparam, mem in zip(
        [Wxh, Whh, Why, bh, by],
        [dWxh, dWhh, dWhy, dbh, dby],
        [mWxh, mWhh, mWhy, mbh, mby]
    ):

        mem += dparam * dparam

        param += (
            -learning_rate
            * dparam
            / np.sqrt(mem + 1e-8)
        )

    # Move forward in text
    p += seq_length

    # =====================================================
    # PRINT TRAINING PROGRESS
    # =====================================================

    if n % 1000 == 0:

        print("=" * 60)
        print(f"Iteration : {n}")
        print(f"Loss      : {loss:.4f}")

        # Generate sample text
        sample_text = sample(
            hprev,
            inputs[0],
            200
        )

        print("\nGenerated Text:\n")
        print(sample_text)
        print("\n")

    n += 1

print("\nTraining Finished.")

Total Characters : 4780
Vocabulary Size  : 73

Training Started...

Iteration : 0
Loss      : 214.5003

Generated Text:

HWItL*mc38ZoNb
JüzqJB50oof1’r0aS*.YL8aw(mB49’ieG6IsGr0lAhw)6PAL’32AqKZJE4uLtGvnB3DM
 gU4.H9hP:eM xLULjh9)*ü,HtJjPlKGHK:EMg09C
cz5cL?Rtlz YLknkC
ardR
c,Y5z)2;ü-e’KzjN(5S)KöIM5nCzuO2t ?85bD99aOxg41Fjoe:


Iteration : 1000
Loss      : 111.6735

Generated Text:

ea haziit ios on til tadoyan 
fisy the tof insddek ho ht rat sh tiz9iciyedcion aEd di*nwor tirdhey tiy the xanlem and,.


ftoond and in 1N3irety mfpcadig staze rrs ol Puwacn rey tortepmicl asal the ao


Iteration : 2000
Loss      : 101.6537

Generated Text:

roE, She, tuon thelched and he picn,edhe walitihp puolteas os ofith yaTteysmisn werapcis6 nprerad vingnedetiphey ofliny obarety afs tichecw fbmenled oxgreorltoM Paaenl, Sniwls tha lh in phe ry arle wh


Iteration : 3000
Loss      : 83.5385

Generated Text:

 of larecsing sorile. to rocsmans ofsads on Pinive woey in Bnmertgiu’s on the Poorkiec 192h he mans on 

# LSTM

### <span style="color:#0275D8">The Solution: Enter the LSTM Cell State</span>

To fix the vanishing gradient problem, the Long Short-Term Memory (LSTM) architecture splits the memory representation. Instead of relying solely on a single hidden state ($h_t$), an LSTM introduces a second, parallel tracking vector: **The Cell State ($C_t$)**.

Think of the Cell State as a **long-term conveyor belt** running straight through the entire timeline. Information can flow along it completely unchanged unless modified by three mathematically controlled **gating mechanisms**:

* <span style="color:#D9534F">**1. Forget Gate ($f_t$):**</span> Decides what old, irrelevant memory to wipe out from the belt.
* <span style="color:#5CB85C">**2. Input Gate ($i_t$):**</span> Decides what new information from the current token should be added to the belt.
* <span style="color:#F0AD4E">**3. Output Gate ($o_t$):**</span> Decides what parts of this updated internal memory should be revealed to the outside world as the current hidden state ($h_t$).

---

#### <span style="color:#5CB85C">🛠️ Why the Math Changes Everything</span>

In a vanilla RNN, the gradient is forced through continuous matrix **multiplication** ($W_{hh}$), leading to exponential decay. 

In an LSTM, the Cell State is updated primarily via linear **addition**:

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

Because addition distributes gradients evenly during backpropagation ($\frac{\partial}{\partial x}(x + y) = 1$), the error signal can flow freely backward through time without vanishing or exploding.

---

### <span style="color:#0275D8">💻 Implementation: Fully Functional NumPy LSTM</span>

Here is your custom, from-scratch implementation of an LSTM cell using only raw NumPy vector operations:


Would you like me to generate the **backward pass (BPTT)** equations and NumPy code for this LSTM, or do you want to provide your next text block?


In [36]:
import numpy as np

# =========================================================
# 1. DATA
# =========================================================

# (previous data )


chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Total Characters :", len(text))
print("Vocabulary Size  :", vocab_size)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

Total Characters : 4780
Vocabulary Size  : 73


In [37]:
# =========================================================
# 2. HYPERPARAMETERS
# =========================================================
hidden_size = 128      # LSTM memory size
seq_length = 50        # BPTT unroll steps
learning_rate = 0.1    # Increased slightly for LSTM Adagrad
iterations = 20000

# =========================================================
# 3. MODEL PARAMETERS (LSTM Architecture)
# =========================================================
# To make the math cleaner and highly optimized, we combine the weights 
# for all 4 gates (Forget, Input, Cell, Output) into one giant matrix.
# We also concatenate the input (x) and previous hidden state (h) into one vector.
concat_size = hidden_size + vocab_size

# W shape: (4 * 128, 128 + vocab_size)
W = np.random.randn(4 * hidden_size, concat_size) * 0.01
b = np.zeros((4 * hidden_size, 1))

# BEST PRACTICE: Initialize forget gate biases to 1.0 so the network 
# naturally remembers everything at the start of training.
b[0:hidden_size, :] = 1.0

# Hidden -> Output (Same as Vanilla RNN)
Why = np.random.randn(vocab_size, hidden_size) * 0.01
by = np.zeros((vocab_size, 1))

In [38]:
# =========================================================
# 4. MATH HELPERS
# =========================================================
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def sigmoid(x):
    # Clip to prevent overflow in exp
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

# =========================================================
# 5. FORWARD + BACKWARD (BPTT)
# =========================================================
def forward_backward(inputs, targets, hprev, Cprev):
    
    # Caching variables for BPTT
    xs, z, gates, f, i, C_bar, o = {}, {}, {}, {}, {}, {}, {}
    hs, Cs, ys, ps = {}, {}, {}, {}
    
    hs[-1] = np.copy(hprev)
    Cs[-1] = np.copy(Cprev)
    loss = 0
    H = hidden_size

    # =====================================================
    # FORWARD PASS
    # =====================================================
    for t in range(len(inputs)):
        
        xs[t] = np.zeros((vocab_size, 1))
        xs[t][inputs[t]] = 1
        
        # Concatenate previous hidden state and current input
        z[t] = np.vstack((hs[t-1], xs[t]))
        
        # Calculate all 4 gates at once using our giant weight matrix
        gates[t] = np.dot(W, z[t]) + b
        
        # Slice the gates into their 4 specific functions
        f[t] = sigmoid(gates[t][0:H])           # Forget Gate
        i[t] = sigmoid(gates[t][H:2*H])         # Input Gate
        C_bar[t] = np.tanh(gates[t][2*H:3*H])   # Candidate Cell State
        o[t] = sigmoid(gates[t][3*H:4*H])       # Output Gate
        
        # The LSTM Conveyor Belt Updates
        Cs[t] = f[t] * Cs[t-1] + i[t] * C_bar[t]
        hs[t] = o[t] * np.tanh(Cs[t])
        
        # Output logits & Probabilities
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = softmax(ys[t])
        loss += -np.log(ps[t][targets[t], 0])

    # =====================================================
    # BACKWARD PASS (BPTT)
    # =====================================================
    dW, db = np.zeros_like(W), np.zeros_like(b)
    dWhy, dby = np.zeros_like(Why), np.zeros_like(by)
    
    dhnext = np.zeros_like(hs[0])
    dCnext = np.zeros_like(Cs[0])

    for t in reversed(range(len(inputs))):
        
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1

        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        
        # Gradient entering the hidden state
        dh = np.dot(Why.T, dy) + dhnext
        
        # Backprop through hidden state -> Output gate
        do = dh * np.tanh(Cs[t])
        do_raw = do * o[t] * (1 - o[t])
        
        # Gradient entering the Cell state
        dC = dh * o[t] * (1 - np.tanh(Cs[t])**2) + dCnext
        
        # Backprop through Cell state -> Forget, Input, and Candidate gates
        df = dC * Cs[t-1]
        df_raw = df * f[t] * (1 - f[t])
        
        di = dC * C_bar[t]
        di_raw = di * i[t] * (1 - i[t])
        
        dC_bar = dC * i[t]
        dC_bar_raw = dC_bar * (1 - C_bar[t]**2)
        
        # Recombine gradients for our giant weight matrix
        dgates = np.vstack((df_raw, di_raw, dC_bar_raw, do_raw))
        
        dW += np.dot(dgates, z[t].T)
        db += dgates
        
        # Pass gradient backward down the sequence
        dz = np.dot(W.T, dgates)
        dhnext = dz[:H]                     # Gradient for the previous hidden state
        dCnext = dC * f[t]                  # Gradient for the previous cell state (The magic step!)

    # Gradient Clipping
    for dparam in [dW, db, dWhy, dby]:
        np.clip(dparam, -5, 5, out=dparam)

    return loss, dW, db, dWhy, dby, hs[len(inputs) - 1], Cs[len(inputs) - 1]

# =========================================================
# 6. TEXT SAMPLING
# =========================================================
def sample(h, C, seed_ix, n):
    x = np.zeros((vocab_size, 1))
    x[seed_ix] = 1
    generated_chars = []
    H = hidden_size

    for _ in range(n):
        z = np.vstack((h, x))
        gates = np.dot(W, z) + b
        
        f = sigmoid(gates[0:H])
        i = sigmoid(gates[H:2*H])
        C_bar = np.tanh(gates[2*H:3*H])
        o = sigmoid(gates[3*H:4*H])
        
        C = f * C + i * C_bar
        h = o * np.tanh(C)
        
        y = np.dot(Why, h) + by
        p = softmax(y)
        
        ix = np.random.choice(range(vocab_size), p=p.ravel())
        x = np.zeros((vocab_size, 1))
        x[ix] = 1
        generated_chars.append(ix_to_char[ix])

    return ''.join(generated_chars)

# =========================================================
# 7. ADAGRAD MEMORY & LOOP
# =========================================================
mW = np.zeros_like(W)
mb = np.zeros_like(b)
mWhy = np.zeros_like(Why)
mby = np.zeros_like(by)

n, p = 0, 0
hprev = np.zeros((hidden_size, 1))
Cprev = np.zeros((hidden_size, 1))

print("\nLSTM Training Started...\n")

while n < iterations:
    if p + seq_length + 1 >= len(text) or n == 0:
        hprev = np.zeros((hidden_size, 1))
        Cprev = np.zeros((hidden_size, 1))
        p = 0

    inputs = [char_to_ix[ch] for ch in text[p:p + seq_length]]
    targets = [char_to_ix[ch] for ch in text[p + 1:p + seq_length + 1]]

    loss, dW, db, dWhy, dby, hprev, Cprev = forward_backward(inputs, targets, hprev, Cprev)

    for param, dparam, mem in zip([W, b, Why, by], 
                                  [dW, db, dWhy, dby], 
                                  [mW, mb, mWhy, mby]):
        mem += dparam * dparam
        param += (-learning_rate * dparam) / np.sqrt(mem + 1e-8)

    p += seq_length

    if n % 1000 == 0:
        print("=" * 60)
        print(f"Iteration : {n} | Loss: {loss:.4f}")
        sample_text = sample(hprev, Cprev, inputs[0], 200)
        print(f"\nGenerated Text:\n{sample_text}\n")
    n += 1


LSTM Training Started...

Iteration : 0 | Loss: 214.5231

Generated Text:
Oüfy’gNgMcdEWDt(E0jyp6EiiS0tiLE,K’FhEwRdLgwf?s4 ’,R-5iwwAmp*nYetmGYvMUSsd6s5gEaJpl*4laCrooPja-a3yscec3kYks)87tqcHhvJst,U*lGr-vikS r.h:2L1Lmriüb)üHEndiIE)yr l:UWtR:e0tIlfDPsmt;8mrTe
z t2els**sPuk
rkWRl

Iteration : 1000 | Loss: 50.3100

Generated Text:
omorting for Elnceamvishe.

D4ring the wossation on thill.

Anterh and as theorice to a sony in the ssied ti stand rediation the war sopvxttic gtnich as rellitivilyy, ad Provecsorsomponted theery by a

Iteration : 2000 | Loss: 43.1055

Generated Text:
his lecal tony and, end his edpollived Amork, in Medaltember in the Faand to fieldinged hom 1520t, and in Brilnt, and thee postiun, Ametication of his was legledicalized his game. In 1901 he wesitual 

Iteration : 3000 | Loss: 10.6562

Generated Text:
 declinered. le conshivation of his year ie fiomieved to wis ptinge.

Ated 1914 and mosy al Warat of Lsteta.ed to thay work in 1911 Professor of the Lolsoinc Physical hi